# Estimación del tamaño muestral necesario

### Funciones

In [ ]:
import math
from scipy.stats import norm

In [ ]:
# Tamaño de muestra
def calcular_tamano_muestra(proporcion_estimada, margen_error, nivel_confianza):
    """Calcula el tamaño de muestra necesario para una proporción."""
    if not (0 <= proporcion_estimada <= 1):
        raise ValueError("La precisión estimada debe estar entre 0 y 1.")
    if not (0 < margen_error < 1):
        raise ValueError("El margen de error debe estar entre 0 y 1.")
    if not (0 < nivel_confianza < 1):
        raise ValueError("El nivel de confianza debe estar entre 0 y 1.")

    # Cálculo dinámico del Z-score usando scipy
    z = norm.ppf(1 - (1 - nivel_confianza) / 2)

    p = proporcion_estimada
    e = margen_error
    n = (z**2 * p * (1 - p)) / (e**2)
    return math.ceil(n)

In [ ]:
# Métricas
def calcular_precision(tp, fp):
    """Calcula la precisión: TP / (TP + FP)"""
    if tp + fp == 0:
        return 0.0
    return tp / (tp + fp)

def calcular_recall(tp, fn):
    """Calcula el recall: TP / (TP + FN)"""
    if tp + fn == 0:
        return 0.0
    return tp / (tp + fn)

def calcular_accuracy(tp, tn, fp, fn):
    """Calcula la accuracy: (TP + TN) / Total"""
    total = tp + tn + fp + fn
    if total == 0:
        return 0.0
    return (tp + tn) / total

def calcular_f1_score(precision, recall):
    """Calcula el F1-score: 2 * (precisión * recall) / (precisión + recall)"""
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

In [ ]:
def calc_metrics(tp, tn, fp, fn):
    # Métricas iniciales
    precision = calcular_precision(tp, fp)
    recall = calcular_recall(tp, fn)
    accuracy = calcular_accuracy(tp, tn, fp, fn)
    f1 = calcular_f1_score(precision, recall)

    print(f"\nMétricas iniciales:")
    print(f"- Precisión: {precision:.4f}")
    print(f"- Recall: {recall:.4f}")
    print(f"- Accuracy: {accuracy:.4f}")
    print(f"- F1-score: {f1:.4f}")
    print()
    
    return precision, recall, accuracy, f1

In [ ]:
def calcular_tamano_muestra_dinamico(precision_observada: float,
                                     N: int = None,
                                     confianzas_deseadas: dict[float] = None,
                                     tolerancias_deseadas: dict[float] = None):
    """
    Calcula el tamaño de muestra necesario para una proporción con ajuste dinámico.
    Si N es None, se calcula el tamaño de muestra sin ajuste.
    Si N es un número entero, se ajusta el tamaño de muestra para una población finita.
    
    Args:
        precision_observada (float): Proporción observada.
        N (int, optional): Tamaño de la población. Si es None, no se ajusta.
    """
    # Listas de escenarios para evaluación
    if confianzas_deseadas == None: confianzas_deseadas = [0.80, 0.90, 0.95, 0.99]
    if tolerancias_deseadas == None: tolerancias_deseadas = [0.01, 0.03, 0.05, 0.10]


    print(f"Ajuste de tamaño de muestra para población finita: para N = {N:.2f}\n") if N else None

    for nivel_confianza in confianzas_deseadas:
        for margen_error in tolerancias_deseadas[::-1]:
            try:
                n = calcular_tamano_muestra(precision_observada, margen_error, nivel_confianza)
                print(f"Confianza deseada: {nivel_confianza*100:.0f}%, Margen de error: ±{margen_error*100:.0f}% -> Tamaño de muestra: {n}")

                if N:
                    n_ajustado = ajuste_por_area(n, N)
                    print(" "*45, f"-> Tamaño ajustado: {n_ajustado}")

            except ValueError as e:
                print(f"Error: {e}")
        print()

In [ ]:
def area_por_altura(altura:float = 3.5):
    resolucion = {
        'Altura': [3, 3.5, 5],
        'Resolucion': [0.070, 0.082, 0.120],
        'Area': [9.8092760003, 13.3515145559, 28.7450726400]
    }

    if altura not in resolucion['Altura']:
        raise ValueError(f"❕No se ha definido el área cubierta para la altura {altura}m.\nAlturas disponibles: {resolucion['Altura']}")

    area = resolucion['Area'][resolucion['Altura'].index(altura)]
    print(f"El área cubierta por la fotografía a {altura}m es de {area:.2f}m².")

    return area

def ajuste_por_area(n, N):
    """
    Aplica la corrección por población finita al tamaño de muestra.
    
    n: tamaño de muestra inicial (sin corrección)
    N: tamaño total de la población
    
    Retorna: tamaño de muestra ajustado
    """
    if N <= 0:
        raise ValueError("La población total N debe ser mayor que 0.")
    return math.ceil(n / (1 + ((n - 1) / N))) # Redondeo ascendente

## Desarrollo

**Fórmula de tamaño de muestra para estimar una proporción poblacional**

In [28]:
from IPython.display import display, Math
display(Math(r'n = \frac{Z^2 \cdot p \cdot (1 - p)}{e^2}'))

<IPython.core.display.Math object>

Donde:

- \( n \): Tamaño de muestra necesario  
- \( Z \): Valor Z correspondiente al nivel de confianza deseado (por ejemplo, 1.96 para 95%)  
- \( p \): Proporción estimada (por ejemplo, precisión esperada del modelo)  
- \( e \): Margen de error permitido (por ejemplo, 0.05 para ±5%)

[*Referencia*](https://chatgpt.com/share/67fc0f7b-b690-8009-b3c2-3940e5654441)

In [33]:
from IPython.display import display, Math
display(Math(r'n = \frac{Z^2 \cdot p \cdot (1 - p)}{e^2}'))

<IPython.core.display.Math object>

## Cálculo por Accuracy

In [35]:
tp = 0.4
tn = 0.
fp = 0.3
fn = 0.3

In [36]:
precision, recall, accuracy, f1 = calc_metrics(tp, tn, fp, fn)
calcular_tamano_muestra_dinamico(precision)


Métricas iniciales:
- Precisión: 0.5714
- Recall: 0.5714
- Accuracy: 0.4000
- F1-score: 0.5714

Confianza deseada: 80%, Margen de error: ±10% -> Tamaño de muestra: 41
Confianza deseada: 80%, Margen de error: ±5% -> Tamaño de muestra: 161
Confianza deseada: 80%, Margen de error: ±3% -> Tamaño de muestra: 447
Confianza deseada: 80%, Margen de error: ±1% -> Tamaño de muestra: 4023

Confianza deseada: 90%, Margen de error: ±10% -> Tamaño de muestra: 67
Confianza deseada: 90%, Margen de error: ±5% -> Tamaño de muestra: 266
Confianza deseada: 90%, Margen de error: ±3% -> Tamaño de muestra: 737
Confianza deseada: 90%, Margen de error: ±1% -> Tamaño de muestra: 6626

Confianza deseada: 95%, Margen de error: ±10% -> Tamaño de muestra: 95
Confianza deseada: 95%, Margen de error: ±5% -> Tamaño de muestra: 377
Confianza deseada: 95%, Margen de error: ±3% -> Tamaño de muestra: 1046
Confianza deseada: 95%, Margen de error: ±1% -> Tamaño de muestra: 9408

Confianza deseada: 99%, Margen de error: ±10

## Cálculo por área cubierta

**Fórmula de corrección por población finita**

In [37]:
display(Math(r'n_{\text{ajustado}} = \frac{n}{1 + \left(\frac{n - 1}{N}\right)}'))

<IPython.core.display.Math object>

Donde:

- \( 𝑁 \): número total de imágenes posibles, calculado como:

In [38]:
display(Math(r'N = \frac{\text{superficie total del lote}}{\text{superficie cubierta por cada imagen}}'))

<IPython.core.display.Math object>

In [54]:
display(Math(r'N = \frac{2{,}000{,}000\ \text{m}^2}{{10}\ \text{m}^2/\text{foto}} = 200{,}000\ \text{fotos}'))

<IPython.core.display.Math object>

### Altura 3.5m 

In [70]:
area = area_por_altura() # 3.5m por defecto

El área cubierta por la fotografía a 3.5m es de 13.35m².


In [71]:
lote = 200 * 10_000 # tamaño del lote (hectáreas)
foto = area # superficie cubierta por cada imagen (m2)

In [72]:
N = lote / foto # Tamaño de la población

print(f"Cada foto cubre {foto:,.0f}m². El lote tiene {lote/10_000:,.0f}ha de extensión.")
print(f"Es necesario tomar {N:,.0f} fotos para cubrir el lote completo.")

Cada foto cubre 13m². El lote tiene 200ha de extensión.
Es necesario tomar 149,796 fotos para cubrir el lote completo.


In [73]:
calcular_tamano_muestra_dinamico(precision, N)

Ajuste de tamaño de muestra para población finita: para N = 149795.74

Confianza deseada: 80%, Margen de error: ±10% -> Tamaño de muestra: 9
                                              -> Tamaño ajustado: 9
Confianza deseada: 80%, Margen de error: ±5% -> Tamaño de muestra: 33
                                              -> Tamaño ajustado: 33
Confianza deseada: 80%, Margen de error: ±3% -> Tamaño de muestra: 91
                                              -> Tamaño ajustado: 91
Confianza deseada: 80%, Margen de error: ±1% -> Tamaño de muestra: 819
                                              -> Tamaño ajustado: 815

Confianza deseada: 90%, Margen de error: ±10% -> Tamaño de muestra: 14
                                              -> Tamaño ajustado: 14
Confianza deseada: 90%, Margen de error: ±5% -> Tamaño de muestra: 54
                                              -> Tamaño ajustado: 54
Confianza deseada: 90%, Margen de error: ±3% -> Tamaño de muestra: 150
                     

### Altura 5m 

In [74]:
area = area_por_altura(5) # 3.5m por defecto

El área cubierta por la fotografía a 5m es de 28.75m².


In [75]:
lote = 200 * 10_000 # tamaño del lote (hectáreas)
foto = area # superficie cubierta por cada imagen (m2)

In [76]:
N = lote / foto # Tamaño de la población

print(f"Cada foto cubre {foto:,.0f}m². El lote tiene {lote/10_000:,.0f}ha de extensión.")
print(f"Es necesario tomar {N:,.0f} fotos para cubrir el lote completo.")

Cada foto cubre 29m². El lote tiene 200ha de extensión.
Es necesario tomar 69,577 fotos para cubrir el lote completo.


In [77]:
calcular_tamano_muestra_dinamico(precision, N)

Ajuste de tamaño de muestra para población finita: para N = 69577.14

Confianza deseada: 80%, Margen de error: ±10% -> Tamaño de muestra: 9
                                              -> Tamaño ajustado: 9
Confianza deseada: 80%, Margen de error: ±5% -> Tamaño de muestra: 33
                                              -> Tamaño ajustado: 33
Confianza deseada: 80%, Margen de error: ±3% -> Tamaño de muestra: 91
                                              -> Tamaño ajustado: 91
Confianza deseada: 80%, Margen de error: ±1% -> Tamaño de muestra: 819
                                              -> Tamaño ajustado: 810

Confianza deseada: 90%, Margen de error: ±10% -> Tamaño de muestra: 14
                                              -> Tamaño ajustado: 14
Confianza deseada: 90%, Margen de error: ±5% -> Tamaño de muestra: 54
                                              -> Tamaño ajustado: 54
Confianza deseada: 90%, Margen de error: ±3% -> Tamaño de muestra: 150
                      

---
## Cálculo para otros valores de CM

In [78]:
try:
    default_tp = 40
    default_fp = 30
    default_fn = 30

    tp_str = input(f"Ingrese el porcentaje de True Positives (TP) [por defecto: {default_tp:.0f}]: ")
    fp_str = input(f"Ingrese el porcentaje de False Positives (FP) [por defecto: {default_fp:.0f}]: ")
    fn_str = input(f"Ingrese el porcentaje de False Negatives (FN) [por defecto: {default_fn:.0f}]: ")

    tp_input = True if tp_str else False
    fp_input = True if fp_str else False
    fn_input = True if fn_str else False

    #print(tp_input,fp_input,fn_input)
    tp = float(tp_str or default_tp) / 100
    fp = float(fp_str or default_fp) / 100
    fn = float(fn_str or default_fn) / 100
    tn = 0  # TN siempre es 0

    # Detectar si el usuario modificó algún valor
    tp_modificado = tp_str != ""
    fp_modificado = fp_str != ""
    fn_modificado = fn_str != ""

    # Recalcular los valores restantes si se modificó uno
    if tp_modificado and not (fp_modificado or fn_modificado):
        if not fp_input and not fn_input:
            fp = (1 - tp)/2
            fn = (1 - tp)/2
        elif fp_input:
            fn = 1 - tp - fp
        elif fn_input:
            fp = 1 - tp - fn
        else:
            raise Exception
    elif fp_modificado and not (tp_modificado or fn_modificado):
        if not tp_input and not fn_input:
            tp = (1 - fp)/2
            fn = (1 - fp)/2
        elif tp_input:
            fn = 1 - tp - fp
        elif fn_input:
            tp = 1 - fn - fp
        else:
            raise Exception
    elif fn_modificado and not (tp_modificado or fp_modificado):
        if not tp_input and not fp_input:
            tp = (1 - fn)/2
            fp = (1 - fn)/2
        elif tp_input:
            fp = 1 - tp - fn
        elif fp_input:
            tp = 1 - fn - fp
        else:
            raise Exception
    elif sum([tp_modificado, fp_modificado, fn_modificado]) > 1:
        # Si se modificaron múltiples valores, se asumen los ingresados
        pass  # La validación posterior se encargará de la suma
    elif not (tp_modificado or fp_modificado or fn_modificado):
        # Si no se ingresó nada, usar los valores por defecto
        tp = default_tp / 100
        fp = default_fp / 100
        fn = default_fn / 100
    else:
        raise Exception

    # Validar entradas
    if any(x < 0 or x > 1 for x in [tp, fp, fn]):
        raise ValueError("Los porcentajes deben estar entre 0 y 100.")
    if not abs(tp + fp + fn - 1) < 1e-9:
        if not tp_input:
            tp = 1 - fn - fp
        elif not fn_input:
            fn = 1 - tp - fp
        elif not fp_input:
            fp = 1 - fn - tp
        else:
            raise ValueError(f"La suma de los porcentajes (TP={tp*100:.2f}%, FP={fp*100:.2f}%, FN={fn*100:.2f}%) debe ser igual a 100%.")
    if tp + tn == 0:
        raise ValueError("La suma de True Positives (TP) y True Negatives (TN) no puede ser cero (TN siempre es 0).")
    if fp + fn == 0:
        raise ValueError("La suma de False Positives (FP) y False Negatives (FN) no puede ser cero.")

    # Mostrar matriz de confusión
    print(f"\nMatriz de confusión:")
    print(f"{tp*100:.2f}% | {fp*100:.2f}%")
    print(f"{fn*100:.2f}% | {tn*100:.2f}%")

    precision, recall, accuracy, f1 = calc_metrics(tp, tn, fp, fn)

    lote = 200 * 10_000 # tamaño del lote (hectáreas)
    area = area_por_altura(float(input(f"Seleccione la altura de vuelo (3 / 3.5 / 5): ") or 3.5))
    foto = area # superficie cubierta por cada imagen (m2)
    N = lote / foto # Tamaño de la población
    print(f"Cada foto cubre {foto:,.0f}m². El lote tiene {lote/10_000:,.0f}ha de extensión.")
    print(f"Es necesario tomar {N:,.0f} fotos para cubrir el lote completo.\n")

    calcular_tamano_muestra_dinamico(precision, N)

except ValueError as e:
    print(f"Error: {e}")
except Exception as e:
    print(f"Ocurrió un error inesperado: {e}")


Matriz de confusión:
40.00% | 30.00%
30.00% | 0.00%

Métricas iniciales:
- Precisión: 0.5714
- Recall: 0.5714
- Accuracy: 0.4000
- F1-score: 0.5714

El área cubierta por la fotografía a 3.5m es de 13.35m².
Cada foto cubre 13m². El lote tiene 200ha de extensión.
Es necesario tomar 149,796 fotos para cubrir el lote completo.

Ajuste de tamaño de muestra para población finita: para N = 149795.74

Confianza deseada: 80%, Margen de error: ±10% -> Tamaño de muestra: 41
                                              -> Tamaño ajustado: 41
Confianza deseada: 80%, Margen de error: ±5% -> Tamaño de muestra: 161
                                              -> Tamaño ajustado: 161
Confianza deseada: 80%, Margen de error: ±3% -> Tamaño de muestra: 447
                                              -> Tamaño ajustado: 446
Confianza deseada: 80%, Margen de error: ±1% -> Tamaño de muestra: 4023
                                              -> Tamaño ajustado: 3918

Confianza deseada: 90%, Margen de er

---
# Comparación con métodos tradicionales
- Tolerancia informada: ±5%
- Area de muestreo: 0.25 m²
- Número de muestras: 10

In [59]:
tn = 0
fp = (200*0.05)/200
fn = (200*0.05)/200
tp = round(1 - fp - fn, 2)

In [60]:
print(tp, tn, fp, fn)

0.9 0 0.05 0.05


In [61]:
precision, recall, accuracy, f1 = calc_metrics(tp, tn, fp, fn)


Métricas iniciales:
- Precisión: 0.9474
- Recall: 0.9474
- Accuracy: 0.9000
- F1-score: 0.9474



In [62]:
lote = 200 * 10_000 # tamaño del lote (hectáreas)
foto = 0.25 # superficie cubierta por cada aro (m2)

In [63]:
N = lote / foto # Tamaño de la población

print(f"Cada aro cubre {foto:,.2f}m². El lote tiene {lote/10_000:,.0f}ha de extensión.")
print(f"Es necesario tomar {N:,.0f} muestras para cubrir el lote completo.")

Cada aro cubre 0.25m². El lote tiene 200ha de extensión.
Es necesario tomar 8,000,000 muestras para cubrir el lote completo.


In [64]:
calcular_tamano_muestra_dinamico(precision, N, tolerancias_deseadas=[0.05])

Ajuste de tamaño de muestra para población finita: para N = 8000000.00

Confianza deseada: 80%, Margen de error: ±5% -> Tamaño de muestra: 33
                                              -> Tamaño ajustado: 33

Confianza deseada: 90%, Margen de error: ±5% -> Tamaño de muestra: 54
                                              -> Tamaño ajustado: 54

Confianza deseada: 95%, Margen de error: ±5% -> Tamaño de muestra: 77
                                              -> Tamaño ajustado: 77

Confianza deseada: 99%, Margen de error: ±5% -> Tamaño de muestra: 133
                                              -> Tamaño ajustado: 133



La efectividad de esta medición dependerá de la varianza del cultivo.

Puede tomarse una medida de la varianza analizando las imágenes ya etiquetadas.